# 🤖 Quick Bot Testing Notebook

Test the Onboarding Assistant bot locally without deploying!

**Quick Start:**
1. Run cells 1-5 to set up
2. Use `await ask("your question")` to test

**Features:**
- Full RAG + LLM pipeline testing
- View retrieved documentation chunks
- See timing metrics
- Test retrieval only (no LLM tokens)


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import time
from pathlib import Path

project_root = Path("/workspaces/py/projects/onboarding_mascot")
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(override=True)

# Unset OPENAI_MODEL so we use the code default (gpt-5-mini)
# The ONLY place to change the model should be in config.py
if 'OPENAI_MODEL' in os.environ:
    del os.environ['OPENAI_MODEL']

print("✅ Environment loaded (using code default for model)")


✅ Environment loaded (using code default for model)


In [3]:
from src.core.config import config
from src.core.bot import OnboardingBot

print("✅ Imports successful")
print(f"\nConfig: {config.openai_model} | RAG Top-K: {config.rag_top_k} | Similarity Threshold: {config.rag_similarity_threshold}")


✅ Imports successful

Config: gpt-5-mini | RAG Top-K: 5 | Similarity Threshold: 0.4


In [4]:
print("🤖 Initializing bot...")
bot = OnboardingBot()
print("✅ Bot ready!")


🤖 Initializing bot...
✅ Bot ready!


In [5]:
async def ask(question: str, show_chunks=True, show_timing=True, log_to_bigquery=False):
    """Test a question through full RAG + LLM pipeline.
    
    Args:
        question: The question to ask
        show_chunks: Whether to display retrieved chunks
        show_timing: Whether to display timing metrics
        log_to_bigquery: Whether to log to BigQuery (TEMPORARY FOR TESTING)
    """
    print("="*80)
    print(f"❓ {question}")
    print("="*80)
    
    total_start = time.time()
    
    # Retrieve docs
    print("\n🔍 Retrieving...")
    retrieval_start = time.time()
    rag_result = bot.retriever.retrieve(
        query=question,
        top_k=config.rag_top_k,
        similarity_threshold=config.rag_similarity_threshold
    )
    retrieval_time = (time.time() - retrieval_start) * 1000
    print(f"✅ Retrieved {rag_result.chunks_returned} chunks")
    
    if show_chunks:
        print("\n📚 Chunks:")
        for chunk in rag_result.chunks:
            print(f"\n  #{chunk.rank}: {chunk.tool} - {chunk.section} (sim: {chunk.similarity:.3f})")
            preview = chunk.content[:150] + "..." if len(chunk.content) > 150 else chunk.content
            print(f"  {preview}")
    
    # Call LLM
    print("\n🤖 Calling LLM...")
    llm_start = time.time()
    response_text, system_prompt_with_context, input_tokens, output_tokens, cached_tokens, reasoning_tokens = await bot.process_message(
        message=question,
        rag_result=rag_result,
        thread_ts=None,
        channel=None
    )
    llm_time = (time.time() - llm_start) * 1000
    total_time = (time.time() - total_start) * 1000
    
    # Display
    print("\n💬 Response:")
    print("=" * 80)
    print(response_text)
    print("=" * 80)
    
    if show_timing:
        print(f"\n⏱️  Retrieval: {retrieval_time:.0f}ms | LLM: {llm_time:.0f}ms | Total: {total_time:.0f}ms")
        print(f"📊 Tokens: input={input_tokens} (cached={cached_tokens}), output={output_tokens} (reasoning={reasoning_tokens})")
    
    print()  # Extra line for spacing
    return response_text


def test_retrieval(query: str, top_k: int = 5):
    """Test only retrieval (no LLM call)."""
    print(f"🔍 Query: {query}")
    result = bot.retriever.retrieve(query=query, top_k=top_k, similarity_threshold=config.rag_similarity_threshold)
    print(f"\n✅ {result.chunks_returned} chunks in {result.retrieval_time_ms:.0f}ms\n")
    
    for chunk in result.chunks:
        print(f"#{chunk.rank}: {chunk.tool} - {chunk.section} (sim: {chunk.similarity:.3f})")
        preview = chunk.content[:200] + "..." if len(chunk.content) > 200 else chunk.content
        print(f"  {preview}\n")
    
    return result


print("✅ Functions ready!")
print("\n💡 Usage:")
print("  await ask('your question')")
print("  await ask('question', show_chunks=False)  # cleaner")
print("  test_retrieval('keyword')  # no LLM")


✅ Functions ready!

💡 Usage:
  await ask('your question')
  await ask('question', show_chunks=False)  # cleaner
  test_retrieval('keyword')  # no LLM


In [6]:
await ask("What should I do if my printer prefers a PNG file, but I want to maintain crisp edges for my design exported from Kittl?")


❓ What should I do if my printer prefers a PNG file, but I want to maintain crisp edges for my design exported from Kittl?

🔍 Retrieving...
✅ Retrieved 5 chunks

📚 Chunks:

  #1: Customer Support - Q&A (sim: 1.000)
  User Question: How can I ensure good print quality when exporting a design as a PNG file from Kittl, especially if the printer prefers PNG?
Answer: To...

  #2: Download Settings - How to Use the Tool (sim: 0.974)
  Navigate to the top-right corner of the editor and click Download.
Use the Artboards dropdown menu to select which artboards to export.
Choose the Fil...

  #3: Customer Support - Q&A (sim: 0.604)
  User Question: How can I prevent my designs from appearing pixelated after exporting them from Kittl, especially when using pro rip software?
Answer: ...

  #4: Customer Support - Q&A (sim: 0.582)
  User Question: My designs are pixelating when downloaded, and I'm unsure about resolution settings in Kittl. What are the optimal export settings to a...

  #5: Customer

'*Best steps to keep PNG edges crisp for print*\n\n1. *Start at final size*: Create your artboard at the exact physical size your printer needs so nothing is upscaled later.  \n2. *Set DPI to 300*: In Download, choose PNG and set DPI to 300 — the printing standard Kittl supports.  \n3. *Export large pixel dimensions*: If your printer prefers PNG, export at larger pixel dimensions (example recommended: 4500 x 5400 px) rather than relying on inches × DPI alone.  \n4. *Use Optimize Quality if available*: If you have a Pro or Expert plan, enable *Optimize Quality* before downloading for crisper results.  \n5. *If edges still need improvement*: Use an external upscaler tool to reinforce edges after download.  \n6. *Prefer PNG for transparency and detail*: PNG preserves sharp edges and transparent backgrounds better than JPEG.\n\n<https://cdnp.kittl.com/6c74d212-a931-444e-8fcd-34a5da750087_Create+Download.mp4|Watch the video>\n\nLet me know if you need help with the exact export settings for

In [ ]:
query = "is there shortcuts for grids?"
max_threshold = 0.4

In [ ]:
# 🐛 DEBUG: Check what's in MascotHelpArticles and what scores we're getting

from weaviate.classes.query import MetadataQuery

collection = bot.retriever.client.collections.use("MascotHelpArticles")

# Check collection info
print("📊 MascotHelpArticles Collection Info:")
print("="*80)
total = collection.aggregate.over_all(total_count=True)
print(f"Total objects: {total.total_count}")
print()

# Try a query with NO threshold to see actual similarity scores
print(f"🔍 Testing query {query} with NO threshold:")
print("="*80)

response = collection.query.hybrid(
    query=query,
    alpha=0.8,
    limit=5,
    max_vector_distance=1-max_threshold,
    return_metadata=MetadataQuery(score=True)
)

print(f"\nFound {len(response.objects)} results:\n")
for i, obj in enumerate(response.objects, 1):
    print(f"Score: {obj.metadata.score:.4f}")
    
    print(f"Rank {i}:")
    print(f"  Score: {obj.metadata.score:.4f}")
    print(f"  Tool: {obj.properties.get('tool', 'N/A')}")
    print(f"  Section: {obj.properties.get('section', 'N/A')}")
    print(f"  Content preview: {obj.properties.get('content', '')[:100]}...")
    print()
    

# print("\n🧪 Now testing with pure semantic search (alpha=1.0):")
# print("="*80)

# response2 = collection.query.hybrid(
#     query="what are templates",
#     alpha=1.0,  # Pure semantic
#     limit=5,
#     return_metadata=MetadataQuery(distance=True)
# )

# print(f"\nFound {len(response2.objects)} results:\n")
# for i, obj in enumerate(response2.objects, 1):
#     distance = obj.metadata.distance if obj.metadata.distance is not None else 1.0
#     similarity = 1.0 - distance
    
#     print(f"Rank {i}:")
#     print(f"  Similarity: {similarity:.4f}")
#     print(f"  Tool: {obj.properties.get('tool', 'N/A')}")
#     print(f"  Section: {obj.properties.get('section', 'N/A')}")
#     print()


📊 OnboardingDocs Collection Info:
Total objects: 88

🔍 Testing query is there shortcuts for grids? with NO threshold:

Found 1 results:

Score: 1.0000
Rank 1:
  Score: 1.0000
  Tool: Keyboard Shortcuts
  Section: Keyboard Shortcuts
  Content preview: Canvas and Navigation
Spacebar – Hold to move the canvas
+ – Zoom in
- – Zoom out
Shift + 0 – Center...

